In [1]:
#ao executar este código, instalamos o suporte ao AMPL no notebook
!pip install -q amplpy
from amplpy import tools
ampl = tools.ampl_notebook(
    modules=["highs", "coin"], # pick from over 20 modules including most commercial and open-source solvers
    license_uuid="bcc3d88a-8b8c-4c93-8f21-c2bedd3fc48f") # your license UUID

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 16.4 MB/s eta 0:00:00
Licensed to AMPL Community Edition License for <santi.everton@gmail.com>.


# Modelagem e implementação de problemas

> **Limitação:** Usem apenas funções lineares para produzir modelos, pois problemas não-lineares possuem algumas particularidas, especialmente quanto à resolução. Neste momento da disciplina, isto pode ser um fator de dificuldade para que a gente possa fazer experimentos.

## Problema 1

**Designação de tarefas**

Uma empresa terceirizada, responsável pela limpeza do prédio da Escola de Ciências e Tecnologia da UFRN, precisa decidir quais de seus funcionários serão responsáveis por cada uma das tarefas de higienização do prédio.

Sabe-se que a tarefa de limpeza é um serviço que apresenta uma certa demanda física, logo a empresa quer dividir estas tarefas de modo que elas sejam realizadas no menor tempo possível a fim de permitir que os funcionários tenham um maior tempo de descanso. Além disso, a redução no tempo de limpeza dos ambientes aumenta o tempo de disponibilidade destes para os usuários.

A empresa contratou $n$ funcionários, pois é necessária a limpeza de $n$ locais diariamente. Neste sentido, chamaremos a ação de **limpar de um ambiente** de "tarefa".

Com base em uma avaliação prévia do ritmo de trabalho de cada funcionário $i~ (i=1, 2, ..., n)$, estimou-se quanto tempo este leva para concluir uma tarefa $j~(j=1, 2, ..., n)$. Este tempo é representado por $t_{ij}$, para todo $i,j=1, 2, ..., n$. Deve-se designar um responsável para cada tarefa.

> **Atividade:** formule o problema descrito usando a forma compacta. Faça também a implementação do mesmo em AMPL para verificar a solução considerando os dados apresentados na tabela abaixo. A tabela mostra o tempo estimado para cada funcionário realizar cada uma das tarefas.


|        ---       | Tarefa 1 | Tarefa 2  | Tarefa 3 | Tarefa 4 |
|     ---       |    ---   |    ---    |    ---   |    ---   |
| Funcionário 1 |        6 |        3  |        2 |        4 |
| Funcionário 2 |       10 |        6  |        2 |        5 |
| Funcionário 3 |        6 |        10 |        9 |        8 |
| Funcionário 4 |       11 |        5  |        4 |        9 |

## Modelo
> **Vamos modelar o problema durante a aula.**

Parâmetros:

* $n$ - Quantidade de funcionários;
* $n$ - Quantidade de tarefas;
* $t_{ij}$ - Tempo gasto pelo funcionário $i$ para executar a tarefa $j$, para todo par $i,j=1,2,\ldots,n$.

Variável de decisão:

* $x_{ij}$, cujo valor deverá ser 1 se o funcionário $i$ é desginado à tarefa $j$, e zero caso contrário, para todo par $i,j=1,2,\ldots,n$;

Função objetivo:

$$
\text{minimize}~Z=~\sum_{i=1}^{n}\sum_{j=1}^{n}t_{ij}x_{ij}
$$

Restrições:

$$
\sum_{j=1}^{n}x_{ij} = 1,~∀~i=1,2,...,n
$$

$$
\sum_{i=1}^{n}x_{ij} = 1,~∀~j=1,2,...,n
$$

$$
x_{ij} \in \{0, 1\}, ∀~i,j=1,2,..., n
$$


Dados retirados de [https://edisciplinas.usp.br/pluginfile.php/4398410/mod_resource/content/0/PO%20II%20-%20M%C3%A3o-de-Obra.pdf](https://edisciplinas.usp.br/pluginfile.php/4398410/mod_resource/content/0/PO%20II%20-%20M%C3%A3o-de-Obra.pdf)





Escreva a formulação aqui

In [22]:
%%writefile tarefas.dat
param n := 4;
param t:    1   2   3   4 :=
        1    6 	 3 	 2 	 4
        2 	10 	 6 	 2 	 5
        3 	 6 	10 	 9 	 8
        4 	11 	 5 	 4 	 9 ;

Overwriting tarefas.dat


In [32]:
%%writefile tarefas.mod
param n > 0;
param t{1..n,1..n} >=0;
var x{1..n,1..n} binary;

minimize Z:sum{i in 1..n, j in 1..n}t[i,j]*x[i,j];

subject to r1{i in 1..n}:sum{j in 1..n}x[i,j]==1;
subject to r2{j in 1..n}:sum{i in 1..n}x[i,j]==1;




Overwriting tarefas.mod


In [33]:
%%writefile tarefas.run
reset;
model "tarefas.mod";
data "tarefas.dat";
option solver highs;
solve;
display Z;
display x;

Overwriting tarefas.run


In [34]:
%%shell
ampl tarefas.run

HiGHS 1.7.0: HiGHS 1.7.0: optimal solution; objective 17
7 simplex iterations
1 branching nodes
Z = 17

x :=
1 1   0
1 2   0
1 3   0
1 4   1
2 1   0
2 2   0
2 3   1
2 4   0
3 1   1
3 2   0
3 3   0
3 4   0
4 1   0
4 2   1
4 3   0
4 4   0
;



## Problema 2

**Formação de equipe Olímpica para o revezamento 4x100 medley**

Ao consultar o site da CBDA (Confederação Brasileira de Desportos Aquáticos) em[https://www.cbda.org.br/natacao/ranking], podemos verificar os tempos dos atletas brasileiros em diversos estilos e distâncias.

Como os tempos dos atletas mudam ao longo do ano esportivo, a CDBA encomendou um programa no qual ela cadastrará os tempos dos 10 melhores atletas em cada estilo para os 100m (Borboleta, Costas, Livre e Peito). O programa então mostrará quais atletas deverão compor a equipe de revezamento e qual estilo nadará, de forma que o tempo previsto de prova da equipe seja mínimo.

> **Atividade:** Formule o problema descrito e resolva usando AMPL.

## Modelo
Descreva o problema e apresente seu modelo. Em seguida, implemente em AMPL para verificar a solução. Imprima na tela o nome do nadador escolhido e qual estilo ele nadará no revezamento. Mostra o tempo previsto de prova.

* Para saber mais sobre o revezamento 4x100 medley, leia [https://www.resumoescolar.com.br/educacao-fisica/regras-da-natacao-estilo-revezamento-medley-4x100-metros/](https://www.resumoescolar.com.br/educacao-fisica/regras-da-natacao-estilo-revezamento-medley-4x100-metros/)





In [ ]:
%%writefile revezamento.mod


Writing revezamento.mod


In [ ]:
%%writefile revezamento.run


Writing revezamento.run


In [ ]:
%%writefile revezamento.dat


Overwriting revezamento.dat


In [ ]:
%%shell
echo "Olá Mundo"

Olá Mundo
